# MediAI — Rule‑Based Prescription & Lifestyle Recommendation (Colab Notebook)

**Purpose:** Rule‑based prototype: prescription OCR → medicine normalization → lifestyle-aware recommendations → JSON output for web integration.

This notebook uses the *Life Style Data* Kaggle dataset and a small canonical medicine CSV. Follow sections in order.

## 1) Setup — install dependencies
Run the following cell to install required Python packages and Tesseract (for OCR).

In [ ]:
# Install packages (may take a minute)
!pip install --quiet transformers sentence-transformers rapidfuzz pandas numpy spacy pillow pytesseract matplotlib

# Install Tesseract engine for OCR
!apt-get update -qq && apt-get install -y -qq tesseract-ocr libtesseract-dev

# Download spaCy small English model (optional)
!python -m spacy download en_core_web_sm
print('Setup complete')

## 2) Mount Google Drive
Place your files in Drive: `life_style_data.csv`, `medicines_metadata.csv`, and a folder with sample prescription images (e.g., `med_samples/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3) Load datasets
Load the Life Style Data and medicines metadata CSVs. Update the paths if you placed files in different Drive folders.

In [ ]:
import pandas as pd
lifestyle_path = '/content/drive/MyDrive/life_style_data.csv'  # update path if needed
meds_path = '/content/drive/MyDrive/medicines_metadata.csv'   # update path if needed

# Load (wrap in try/except so notebook doesn't crash if files are not present)
import os
if os.path.exists(lifestyle_path):
    lifestyle = pd.read_csv(lifestyle_path)
    print('Loaded lifestyle dataset:', lifestyle.shape)
else:
    lifestyle = None
    print('life_style_data.csv not found at', lifestyle_path)

if os.path.exists(meds_path):
    meds = pd.read_csv(meds_path)
    print('Loaded medicines metadata:', meds.shape)
else:
    meds = None
    print('medicines_metadata.csv not found at', meds_path)

## 4) Quick EDA & preprocessing (example)
Adjust the column names and transformations to match your dataset.

In [ ]:
# Example cleaning (customize according to actual column names)
if lifestyle is not None:
    print(lifestyle.columns.tolist())
    # Example: convert an 'Age' column to numeric if present
    if 'Age' in lifestyle.columns:
        lifestyle['age'] = pd.to_numeric(lifestyle['Age'], errors='coerce')
    # Convert smoking to boolean if a 'Smoking' column exists
    if 'Smoking' in lifestyle.columns:
        lifestyle['smoker_flag'] = lifestyle['Smoking'].apply(lambda x: True if str(x).strip().lower() in ['yes','true','1'] else False)
    display(lifestyle.head().style.set_caption('Lifestyle sample (head)'))
else:
    print('No lifestyle dataframe loaded.')

## 5) Starter medicines metadata (create if you don't have one)
This block creates a starter `medicines_metadata.csv` in your Drive for demonstration.

In [ ]:
# Create a starter medicine metadata CSV if it doesn't exist
import pandas as pd, os
starter_path = '/content/drive/MyDrive/medicines_metadata.csv'
if not os.path.exists(starter_path):
    starter_meds = [
        {"canonical_id":"MET500TAB","name":"Metformin","salt":"Metformin","dosage_forms":"Tablet","manufacturer":"XYZ Pharma","mrp_inr":120,"class":"Antidiabetic","notes":"May reduce B12 absorption"},
        {"canonical_id":"ATO10TAB","name":"Atorvastatin","salt":"Atorvastatin","dosage_forms":"Tablet","manufacturer":"ABC Labs","mrp_inr":200,"class":"Statin","notes":"Avoid grapefruit"},
        {"canonical_id":"PAR500TAB","name":"Paracetamol","salt":"Paracetamol","dosage_forms":"Tablet","manufacturer":"MediCare","mrp_inr":30,"class":"Analgesic","notes":"Standard pain reliever"}
    ]
    meds_df = pd.DataFrame(starter_meds)
    meds_df.to_csv(starter_path, index=False)
    print('Starter medicines_metadata.csv created at', starter_path)
else:
    print('medicines_metadata.csv already exists at', starter_path)

# reload meds variable
if os.path.exists(starter_path):
    meds = pd.read_csv(starter_path)
    display(meds.head().style.set_caption('Medicines metadata sample'))

## 6) OCR using pytesseract and simple parsing rules

In [ ]:
from PIL import Image
import pytesseract
import re

STRENGTH_UNIT_PATTERN = r"(\d+\s*(mg|g|mcg|ml|IU)\b)"

def ocr_image_to_text(image_path):
    img = Image.open(image_path).convert('RGB')
    text = pytesseract.image_to_string(img)
    return text

def extract_medicine_lines(ocr_text):
    lines = [ln.strip() for ln in ocr_text.split('\n') if ln.strip()]
    med_lines = []
    for ln in lines:
        if re.search(STRENGTH_UNIT_PATTERN, ln, flags=re.I) or re.search(r"\b(tab|tablet|cap|capsule|syrup|injection)\b", ln, flags=re.I):
            med_lines.append(ln)
    return med_lines

# Demo: specify a sample image path from your Drive
sample_img = '/content/drive/MyDrive/med_samples/sample_prescription_1.jpg'  # change if needed
if os.path.exists(sample_img):
    text = ocr_image_to_text(sample_img)
    print('--- OCR TEXT ---\n', text)
    print('--- Extracted candidate lines ---\n', extract_medicine_lines(text))
else:
    print('Sample image not found at', sample_img, '\nPlace an image in that path to run OCR demo.')

## 7) Medicine normalization (fuzzy matching with RapidFuzz)

In [ ]:
from rapidfuzz import fuzz, process

# Precompute canonical names
if meds is not None:
    canonical_names = meds['name'].tolist()
else:
    canonical_names = []

def clean_token(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9 ]+"," ", s)
    s = re.sub(r"\s+"," ", s).strip()
    return s

def normalize_medicine(candidate_text, canonical_list=canonical_names, scorer=fuzz.token_sort_ratio):
    cand = re.sub(STRENGTH_UNIT_PATTERN, '', candidate_text, flags=re.I)
    cand = re.sub(r"\b(tab|tablet|cap|capsule|syrup|injection|once|twice|daily|bd|od|tds)\b", '', cand, flags=re.I)
    cand_clean = clean_token(cand)
    if not canonical_list:
        return None
    match = process.extractOne(cand_clean, canonical_list, scorer=scorer)
    if match:
        name, score, idx = match
        row = meds[meds['name'] == name].iloc[0].to_dict()
        return {"extracted_text": candidate_text, "candidate_clean": cand_clean, "match_name": name, "score": score, "canonical": row}
    return None

# Example normalization (if OCR produced candidates)
if os.path.exists(sample_img):
    cand_lines = extract_medicine_lines(ocr_image_to_text(sample_img))
    normalized = [normalize_medicine(c) for c in cand_lines]
    from pprint import pprint
    pprint(normalized)
else:
    print('No sample image — normalization demo skipped.')

## 8) Rule-based recommendation engine (simple rules). Extend `CLASS_RULES` for more coverage.

In [ ]:
# Rules mapping
CLASS_RULES = {
    'Statin': {
        'alerts': ['Avoid grapefruit juice — can raise statin levels and increase side effects.'],
        'diet_avoid': ['Grapefruit juice', 'High-fat meals'],
        'diet_include': ['Leafy greens', 'Whole grains']
    },
    'Antidiabetic': {
        'alerts': ['Monitor blood glucose; be careful with high-sugar foods.'],
        'diet_avoid': ['Sugary beverages', 'Refined carbs'],
        'diet_include': ['High-fiber foods', 'Lean protein']
    },
    'Analgesic': {
        'alerts': ['Avoid excessive alcohol while on analgesics.'],
        'diet_avoid': ['Alcohol'],
        'diet_include': ['Hydrating fluids']
    }
}

LIFESTYLE_RULES = {
    'smoker': {
        True: ['Quitting smoking reduces drug interactions and improves recovery.'],
        False: []
    },
    'alcohol': {
        True: ['Avoid alcohol when taking many medicines - increases liver load.'],
        False: []
    }
}

def generate_recommendations(user_profile, normalized_meds):
    rec = {"diet": {"include": set(), "avoid": set()}, "exercise": None, "lifestyle_tips": set(), "alerts": []}

    if user_profile.get('smoker'):
        rec['lifestyle_tips'].update(LIFESTYLE_RULES['smoker'][True])
    if user_profile.get('alcohol'):
        rec['lifestyle_tips'].update(LIFESTYLE_RULES['alcohol'][True])

    for nm in normalized_meds:
        if nm is None: continue
        mclass = nm['canonical'].get('class')
        if mclass and mclass in CLASS_RULES:
            rules = CLASS_RULES[mclass]
            rec['alerts'].extend([{"medicine": nm['canonical']['name'], "alert": a} for a in rules.get('alerts', [])])
            for it in rules.get('diet_include', []): rec['diet']['include'].add(it)
            for it in rules.get('diet_avoid', []): rec['diet']['avoid'].add(it)
            if mclass == 'Antidiabetic':
                rec['exercise'] = 'Moderate exercise 30 minutes daily (walking), monitor glucose.'

    if not rec['exercise']:
        el = user_profile.get('exercise_level', 'moderate')
        if el in ['none','low']:
            rec['exercise'] = 'Start with light walks 15-20 minutes daily; consult physician before intense workouts.'
        else:
            rec['exercise'] = 'Maintain current exercise level; avoid sudden high-intensity workouts when starting new meds.'

    rec['diet']['include'] = list(rec['diet']['include'])
    rec['diet']['avoid'] = list(rec['diet']['avoid'])
    rec['lifestyle_tips'] = list(rec['lifestyle_tips'])
    return rec

# Example usage
example_user = {"age":48, "gender":"Male", "exercise_level":"low", "diet_type":"Non-Vegetarian", "smoker":False, "alcohol":True, "health_conditions":["Type 2 Diabetes"]}
# normalized variable may exist from earlier; if not, set empty
try:
    sample_normalized = normalized
except NameError:
    sample_normalized = []
recs = generate_recommendations(example_user, sample_normalized)
print('Recommendations sample:')
import pprint; pprint.pprint(recs)

## 9) Build JSON output schema and save to Drive

In [ ]:
import json
from datetime import datetime

def build_output_json(user_profile, normalized_meds, recs):
    out = {"user_profile": user_profile, "prescription_analysis": [], "recommendations": recs, "metadata": {"processed_at": datetime.utcnow().isoformat() + 'Z', "version": "v0.1.0"}}
    for nm in normalized_meds:
        if nm is None: continue
        can = nm['canonical']
        out['prescription_analysis'].append({
            "extracted_text": nm['extracted_text'],
            "medicine_name": can['name'],
            "strength": nm.get('strength', ''),
            "form": can.get('dosage_forms',''),
            "match_confidence": nm['score']/100.0,
            "canonical_id": can['canonical_id'],
            "manufacturer": can.get('manufacturer',''),
            "mrp_inr": can.get('mrp_inr', None),
            "availability": can.get('availability','Unknown'),
            "generic_available": True if 'generic' in can.get('notes','').lower() else False
        })
    return json.dumps(out, indent=2)

# Save example JSON if normalized and recs exist
try:
    out_json = build_output_json(example_user, sample_normalized, recs)
    save_path = '/content/drive/MyDrive/med_output_example.json'
    with open(save_path, 'w') as f:
        f.write(out_json)
    print('Saved example JSON to', save_path)
except Exception as e:
    print('Could not build/save JSON (missing variables?):', e)

## 10) Next steps & integration tips
- Add manual confirmation UI for low confidence items.
- Expand CLASS_RULES with more drug classes & specific drug-level rules.
- Automate medicine price updates from Indian pharmacy APIs.
- Wrap inference into a FastAPI service for the website to call `/analyze_prescription` and receive the JSON.
- Replace rule-based parsing with spaCy/BERT NER when you have labeled data.